# 基于梯度提升的 NIPT 最佳孕周推荐模型
目标：使用 年龄 / 身高 / 体重 / Y染色体达标比例 / 孕周 构建风险模型，按 BMI 分组找到潜在风险最小的推荐 NIPT 时点，并做误差敏感性分析。

In [ ]:
import os, re, math, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
sns.set_style('whitegrid')
plt.rcParams['font.sans-serif'] = ['SimHei']; plt.rcParams['axes.unicode_minus'] = False
file_path = '3.xlsx'
if not os.path.exists(file_path):
    raise FileNotFoundError('未找到 3.xlsx，请放置于当前工作目录。')
df = pd.read_excel(file_path)
print('原始数据形状:', df.shape)

## 列自动匹配与清洗

In [ ]:
# 尝试模糊匹配列名
cols_lower = {c: str(c).lower() for c in df.columns}
def find_col(patterns):
    for c in df.columns:
        cl = cols_lower[c]
        if any(p in cl for p in patterns):
            return c
    return None
col_age   = find_col(['年龄'])
col_height= find_col(['身高','身 長'.replace(' ',''),'height','高'])
col_weight= find_col(['体重','weight','重'])
col_yprop = find_col(['y','染色体'])
col_week  = find_col(['孕周','检测孕周','周数'])
risk_candidates = ['风险','风险值','风险评分','是否高风险','胎儿是否健康','风险标记']
col_risk = None
for r in risk_candidates:
    for c in df.columns:
        if r == c or r in str(c):
            col_risk = c; break
    if col_risk: break
print('匹配列: 年龄=', col_age, ' 身高=', col_height, ' 体重=', col_weight, ' Y比例=', col_yprop, ' 孕周=', col_week, ' 风险列=', col_risk)
needed = [col_age,col_height,col_weight,col_yprop,col_week,col_risk]
if any(c is None for c in needed):
    missing = [lab for lab,c in zip(['年龄','身高','体重','Y比例','孕周','风险'], needed) if c is None]
    print('警告: 以下列未匹配成功, 需手动调整代码:', missing)
# 数值化处理
def to_float(s):
    return pd.to_numeric(s, errors='coerce')
for c in [col_age,col_height,col_weight,col_yprop,col_week]:
    if c: df[c] = to_float(df[c])
# 身高单位纠正
if col_height and df[col_height].dropna().median() > 100:  # 认为是 cm
    df[col_height] = df[col_height] / 100.0
# BMI
if col_height and col_weight:
    df['BMI'] = df[col_weight] / (df[col_height]**2)
else:
    df['BMI'] = np.nan
# 风险列构造
if col_risk is None:
    # 尝试用 胎儿是否健康 的反向, 或回退为随机占位(不推荐, 仅保证代码可运行)
    if '胎儿是否健康' in df.columns:
        col_risk = '胎儿是否健康'
    else:
        df['临时风险'] = np.random.binomial(1, 0.3, size=len(df))
        col_risk = '临时风险'
risk_raw = df[col_risk]
if risk_raw.nunique() <= 2:
    # 可能是 1=健康 或 1=高风险 -> 通过列名简单判断
    if any(k in col_risk for k in ['健康']):
        risk = 1 - pd.to_numeric(risk_raw, errors='coerce').fillna(0)  # 健康->0风险
    else:
        risk = pd.to_numeric(risk_raw, errors='coerce').fillna(0)
else:
    risk = pd.to_numeric(risk_raw, errors='coerce')
risk = (risk - risk.min()) / (risk.max() - risk.min() + 1e-9)  # 归一化为 0~1
df['风险值'] = risk
# BMI 分组
def bmi_group(x):
    if pd.isna(x): return '未知'
    if x < 18.5: return '低体重'
    if x < 24: return '正常'
    if x < 28: return '超重'
    return '肥胖'
df['BMI组'] = df['BMI'].apply(bmi_group)
print(df[['BMI','BMI组','风险值']].head())

## 训练梯度提升模型 (回归形式，目标=风险值)

In [ ]:
feature_cols = []
mapping = [('年龄', col_age), ('身高', col_height), ('体重', col_weight), ('Y比例', col_yprop), ('检测孕周', col_week), ('BMI', 'BMI')]
for alias, real in mapping:
    if real and real in df.columns:
        if alias != real:
            df[alias] = df[real]  # 统一别名
        feature_cols.append(alias)
# 去除目标自身潜在泄露列（如果‘风险’相关词在特征中出现则移除）
feature_cols = [c for c in feature_cols if '风险' not in c]
print('特征列:', feature_cols)
model_df = df.dropna(subset=feature_cols + ['风险值']).copy()
X = model_df[feature_cols].values
y = model_df['风险值'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
gbr = GradientBoostingRegressor(random_state=42, n_estimators=400, learning_rate=0.05, max_depth=3)
gbr.fit(X_train, y_train)
pred = gbr.predict(X_test)
print('MSE:', mean_squared_error(y_test, pred), ' R2:', r2_score(y_test, pred))
importances = gbr.feature_importances_
imp_df = pd.DataFrame({'feature': feature_cols, 'importance': importances}).sort_values('importance', ascending=False)
print('特征重要性:\n', imp_df)
plt.figure(figsize= (6,4)); sns.barplot(x='importance', y='feature', data=imp_df, palette='viridis'); plt.title('特征重要性'); plt.tight_layout(); plt.show()

## 按 BMI 组扫描孕周，寻找最优 NIPT 时点

In [ ]:
if col_week is None and '检测孕周' not in df.columns:
    raise RuntimeError('未找到孕周列，无法扫描。')
week_col_alias = '检测孕周'
weeks_valid = model_df[week_col_alias].dropna()
week_min, week_max = math.floor(weeks_valid.min()), math.ceil(weeks_valid.max())
week_grid = np.linspace(week_min, week_max, num=(week_max - week_min + 1))
results = []
plt.figure(figsize=(8,5))
for group, sub in model_df.groupby('BMI组'):
    if sub.empty: continue
    medians = sub[feature_cols].median()
    curve = []
    for w in week_grid:
        sample = medians.copy()
        if week_col_alias in sample.index:
            sample[week_col_alias] = w
        else:
            # 若孕周列原名不同，已在 feature_cols 中别名为 '检测孕周'
            sample.loc[feature_cols.index(week_col_alias)] = w
        risk_pred = gbr.predict([sample.values])[0]
        curve.append(risk_pred)
    curve = np.array(curve)
    best_idx = curve.argmin()
    best_week = week_grid[best_idx]
    results.append({'BMI组': group, '最优孕周': float(best_week), '最小风险': float(curve.min())})
    plt.plot(week_grid, curve, label=f'{group} (最佳{best_week:.1f}周)')
plt.xlabel('孕周'); plt.ylabel('预测风险'); plt.title('不同 BMI 组孕周-风险曲线'); plt.legend(); plt.tight_layout(); plt.show()
opt_df = pd.DataFrame(results).sort_values('最优孕周')
print('最优孕周表:')
print(opt_df.to_string(index=False))

## 误差敏感性分析
对输入特征(除孕周)加噪声，重复模拟，观察“最优孕周”稳定性。

In [ ]:
rng = np.random.default_rng(123)
n_iter = 150  # Monte Carlo 次数
noise_scale = 0.1  # 噪声比例(相对各特征标准差)
week_opt_samples = []  # [(组, 最优孕周), ...]
feat_std = model_df[feature_cols].std()
for group, sub in model_df.groupby('BMI组'):
    if sub.empty: continue
    medians = sub[feature_cols].median()
    for _ in range(n_iter):
        perturbed = medians.copy()
        for fc in feature_cols:
            if fc == week_col_alias: continue
            perturbed[fc] += rng.normal(0, feat_std[fc] * noise_scale)
        curve = []
        for w in week_grid:
            p = perturbed.copy(); p[week_col_alias] = w
            curve.append(gbr.predict([p.values])[0])
        curve = np.array(curve)
        week_opt_samples.append({'BMI组': group, '最优孕周': week_grid[curve.argmin()]})
sens_df = pd.DataFrame(week_opt_samples)
plt.figure(figsize=(7,4))
sns.boxplot(data=sens_df, x='BMI组', y='最优孕周')
plt.title('最优孕周分布(噪声扰动)'); plt.tight_layout(); plt.show()
plt.figure(figsize=(8,5))
for group, sub in sens_df.groupby('BMI组'):
    sns.kdeplot(sub['最优孕周'], label=group, fill=True, alpha=0.3)
plt.title('最优孕周概率密度(扰动)'); plt.xlabel('孕周'); plt.legend(); plt.tight_layout(); plt.show()
stab = sens_df.groupby('BMI组')['最优孕周'].agg(['mean','std']).reset_index().rename(columns={'mean':'均值','std':'标准差'})
print('稳定性统计:\n', stab.to_string(index=False))

## 结果导出（可选）

In [ ]:
opt_df.to_csv('最优孕周结果.csv', index=False)
print('已保存: 最优孕周结果.csv')